# 03_tool_calling_and_mcp: Pydantic Validation & Stdio MCP Server Connection

This notebook creates an MCP Server with user database lookup tools, and connects an MCP Client to execute schemas using Pydantic and Instructor validation.


In [1]:
import os
import sys
import asyncio
from dotenv import load_dotenv
from pydantic import BaseModel, Field

# 1. Write the mcp_server.py helper script to disk
server_code = r'''# Simple stdio MCP Server
import asyncio
from mcp.server.models import InitializationOptions
from mcp.server import Server
import mcp.types as types
from mcp.server.stdio import stdio_server

server = Server("user-db-server")

@server.list_tools()
async def handle_list_tools():
    return [
        types.Tool(
            name="lookup_user",
            description="Look up user records by name.",
            inputSchema={
                "type": "object",
                "properties": {
                    "name": {"type": "string"}
                },
                "required": ["name"]
            }
        )
    ]

@server.call_tool()
async def handle_call_tool(name: str, arguments: dict):
    if name == "lookup_user":
        username = arguments.get("name")
        return [types.TextContent(type="text", text=f"User database record: {username}, Role=AI_Engineer, Salary=$150,000")]
    raise ValueError(f"Tool {name} not found")

async def main():
    async with stdio_server() as (read_stream, write_stream):
        await server.run(
            read_stream,
            write_stream,
            InitializationOptions(
                server_name="user-db-server",
                server_version="1.0.0",
                capabilities=types.ServerCapabilities(
                    tools=types.ToolsCapability()
                )
            )
        )

if __name__ == "__main__":
    asyncio.run(main())
'''

with open("mcp_server.py", "w", encoding="utf-8") as f:
    f.write(server_code)

print("Saved mcp_server.py helper script.")


Saved mcp_server.py helper script.


In [2]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# 2. Connect client to stdio server
async def run_client():
    server_params = StdioServerParameters(
        command=sys.executable,
        args=["mcp_server.py"]
    )
    
    with open("mcp_err.log", "w", encoding="utf-8") as err_f:
        async with stdio_client(server_params, errlog=err_f) as (read_stream, write_stream):
            async with ClientSession(read_stream, write_stream) as session:
                await session.initialize()
                print("Connected to MCP Server!")
                
                # List tools
                tools = await session.list_tools()
                print("Exposed Tools:", [t.name for t in tools.tools])
                
                # Call tool
                result = await session.call_tool("lookup_user", {"name": "Alice"})
                print("Call Result:", result.content[0].text)

# Run the client loop inside the notebook event loop
await run_client()



Connected to MCP Server!
Exposed Tools: ['lookup_user']
Call Result: User database record: Alice, Role=AI_Engineer, Salary=$150,000


### Output Explanation
- The stdio MCP Server is written and executed using the virtual environment interpreter `sys.executable`.
- The MCP Client initiates connection handshakes, dynamically queries available schemas, and triggers the `lookup_user` tool execution successfully.
